# BIOMASS L1C STA — stream two acquisitions, NetCDF stack and Pauli RGB

**No local BIOMASS product downloads required.** This version follows **Biomass Data Access Example.ipynb**: query the ESA MAAP STAC collection, read `enclosure_i_abs_tiff` / `enclosure_i_phase_tiff` assets using your bearer token and `fsspec` + `rasterio`, and reconstruct HH/HV/VH/VV.

Run your original data-access notebook through the cell that defines `items` and `access_token`, **in the same Jupyter kernel**, before running this one. Alternatively, configure the token in the optional cell below. The only locally written files are the **output** NetCDF and figures.

**Note:** Stack comparison is valid only for co-registered STA acquisitions from the same `eofeos:stack_id`. This notebook checks that ID, raster shape and band counts, but cannot independently prove pixel registration.

## 1. Imports and output settings

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rasterio as rio
from rasterio.windows import Window
import fsspec
from pystac_client import Client

OUTPUT_FOLDER = Path("biomass_outputs")  # outputs only; no STA_FOLDER needed
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
WINDOW_SIZE = (1024, 1024)  # (height, width) pixels; set None for full rasters (large RAM)
PAULI_LOOKS = (11, 3)       # azimuth x range boxcar smoothing
POLARIZATIONS = ("HH", "HV", "VH", "VV")


## 2. Use catalog results from your existing notebook

If `items` already exists from **Biomass Data Access Example.ipynb**, it is reused. Otherwise this cell repeats your *Wilkins Ice Shelf* STAC query. Your original dates and bbox are preserved. Check the returned stack IDs before choosing acquisitions.

In [ ]:
if "items" not in globals():
    catalog = Client.open("https://catalog.maap.eo.esa.int/catalogue/")
    search = catalog.search(
        collections=["BiomassLevel1c"],
        filter=("product:type='S1_STA__1S' or "
                "product:type='S2_STA__1S' or "
                "product:type='S3_STA__1S'"),
        datetime=["2025-01-01T00:00:00Z", "2026-10-13T23:59:59Z"],
        # bbox=[-75.1451, -71.5351, -69.0196, -69.4491],  # wilkins
        bbox = [7.272949,-71.566100,13.996582,-69.660815], # nivilsen
        method="GET", max_items=320,
    )
    items = list(search.items())

print(f"Found {len(items)} catalog items")
for i, it in enumerate(items[:30]):
    print(i, it.id, "stack:", it.properties.get("eofeos:stack_id"),
          "datetime:", it.properties.get("datetime"))


## 3. Choose two co-registered products

By default, select the first *two distinct* STA catalog items with the same nonempty stack ID and both TIFF asset URLs. To override, set `SELECTED_INDICES = (i, j)` from the printed list and rerun. An empty stack ID does **not** establish coregistration.

In [ ]:
# selected_indices = [7, 285, 37, 318, 163] # wilkins

selected_indices = [134] # nivlisen
#selected_indices = [224]   # george vi

for idx in selected_indices:
    item = items[idx]
    print(
        f"Index: {idx} | "
        f"Stack: {item.properties.get('eofeos:stack_id')} | "
        f"Product: {item.id}"
    )

In [ ]:
from collections import defaultdict

stacks = defaultdict(list)

for idx, item in enumerate(items):
    stack_id = item.properties.get("eofeos:stack_id")

    if stack_id is not None:
        stacks[stack_id].append(idx)

print(f"Total stacks: {len(stacks)}")

for stack_id, indices in stacks.items():
    print(f"Stack {stack_id}: {indices}")

In [ ]:
for stack_id, indices in list(stacks.items())[:3]:
    print(f"\nSTACK: {stack_id}")

    for idx in indices:
        item = items[idx]

        print(
            f"Index: {idx:3d} | "
            f"Start: {item.properties.get('start_datetime')} | "
            f"End: {item.properties.get('end_datetime')}"
        )

In [ ]:
from pathlib import Path
from BPS_inputs_download import get_stack

# Your desired product indices
# SELECTED_INDICES = [224]

SELECTED_INDICES = [134]  # nivlisen updated

INPUT_FOLDER = Path("biomass_inputs")
INPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Find the stacks containing these indices
selected_stacks = {}

for stack_id, indices in stacks.items():
    matching = set(SELECTED_INDICES).intersection(indices)

    if matching:
        selected_stacks[stack_id] = indices
        print(f"Stack ID: {stack_id}")
        print(f"  Selected products: {sorted(matching)}")
        print(f"  All products: {indices}")

# Download each unique stack
for stack_id, indices in selected_stacks.items():

    reference_id = items[indices[0]].id

    # Use actual eofeos:stack_id as folder name
    stack_folder = INPUT_FOLDER / str(stack_id)
    stack_folder.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading Stack: {stack_id}")

    downloaded_items = get_stack(
        reference_id,
        str(stack_folder)
    )

    print(f"Completed Stack: {stack_id}")

## 4. Authorization

Use `access_token` already obtained in the original notebook; do **not** copy client secrets or offline tokens into this notebook. If running separately, set the environment variable `MAAP_ACCESS_TOKEN` to a **valid short-lived bearer access token**. The token must have the appropriate MAAP data permissions.

In [ ]:
import os
access_token = globals().get("access_token") or os.environ.get("MAAP_ACCESS_TOKEN")
if not access_token:
    raise RuntimeError("No access_token. Run the authorization cells of Biomass Data Access Example.ipynb first, or set MAAP_ACCESS_TOKEN.")
print("Bearer token available (contents hidden).")


## 5. Stream amplitude and phase windows — no TIFF downloads

Implements the same `fsspec.filesystem('https', headers=...)` + `rasterio.open(stream)` strategy as your original notebook. A window reduces *array memory*; depending on the remote server/fsspec backend, HTTP I/O may still involve caching substantial content. Default window is top-left 1024×1024; change `WINDOW_SIZE` or set a custom `Window` for a specific area.

In [ ]:
def read_remote_bands(url, token, window=None):
    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
    with fs.open(url, "rb") as stream:
        with rio.open(stream) as src:
            shape = (src.height, src.width)
            count = src.count
            bands = src.read(window=window, masked=True)
            descriptions = src.descriptions
    return bands, shape, count, descriptions

def load_complex_product(item, token, window):
    amp, ashape, acount, adesc = read_remote_bands(
        item.assets["enclosure_i_abs_tiff"].href, token, window)
    phase, pshape, pcount, pdesc = read_remote_bands(
        item.assets["enclosure_i_phase_tiff"].href, token, window)
    if ashape != pshape or acount != pcount:
        raise ValueError(f"Amplitude/phase mismatch for {item.id}")
    if acount != 4:
        raise ValueError(f"Expected 4 TIFF bands HH,HV,VH,VV, found {acount} for {item.id}. Inspect TIFF band descriptions {adesc}.")
    # Your notebook maps 4 bands in this order: HH, HV, VH, VV.
    # Confirm against product metadata if the sensor/product representation changes.
    if any(d is not None for d in adesc):
        print(item.id, "amplitude band descriptions:", adesc)
    arr = np.asarray(amp.filled(np.nan), dtype=np.float32) * np.exp(
        1j * np.asarray(phase.filled(np.nan), dtype=np.float32))
    return arr.astype(np.complex64), ashape

# Determine common pixel window from metadata via a tiny initial read.
first_url = chosen[0].assets["enclosure_i_abs_tiff"].href
_, native_shape, _, _ = read_remote_bands(first_url, access_token, Window(0, 0, 1, 1))
if WINDOW_SIZE is None:
    window = None
else:
    h = min(WINDOW_SIZE[0], native_shape[0])
    w = min(WINDOW_SIZE[1], native_shape[1])
    window = Window(0, 0, w, h)
print("Native raster shape:", native_shape, "| selected window:", window)


## 6. Build 2-acquisition polarization stack

In [ ]:
acquisitions = []
for it in chosen:
    slc, shape = load_complex_product(it, access_token, window)
    if shape != native_shape:
        raise ValueError(f"Not the same raster shape: {shape} vs {native_shape}")
    acquisitions.append(slc)

slc_stack = np.stack(acquisitions, axis=0)  # (acquisition, polarization, azimuth, range)
print("Stack shape:", slc_stack.shape)
print("Polarizations:", POLARIZATIONS)


## 7. Export NetCDF (`.nc`)

Store the real and imaginary components separately for broad NetCDF compatibility. Data dimensions are pixel-indexed (`azimuth`, `range`) and **not latitude/longitude**. Disk size grows quickly at full resolution.

In [ ]:
_, _, H, W = slc_stack.shape
ds = xr.Dataset(
    data_vars={
        "slc_real": (("acquisition", "polarization", "azimuth", "range"), slc_stack.real.astype("float32")),
        "slc_imag": (("acquisition", "polarization", "azimuth", "range"), slc_stack.imag.astype("float32")),
    },
    coords={
        "acquisition": [0, 1], "polarization": list(POLARIZATIONS),
        "azimuth": np.arange(H), "range": np.arange(W),
        "product_id": ("acquisition", [it.id for it in chosen]),
    },
    attrs={"stack_id": str(stack_ids[0]), "description": "MAAP BIOMASS STA streamed complex SLC subset; pixel-indexed, not geocoded"},
)
nc_path = OUTPUT_FOLDER / "biomass_stack.nc"
ds.to_netcdf(nc_path, engine="netcdf4", encoding={
    "slc_real": {"zlib": True, "complevel": 3},
    "slc_imag": {"zlib": True, "complevel": 3},
})
print("Wrote", nc_path)
display(ds)


## 8. Visualize four polarization intensities (dB)

In [ ]:
for a in range(slc_stack.shape[0]):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    for p, (ax, name) in enumerate(zip(axes, POLARIZATIONS)):
        power_db = 10 * np.log10(np.maximum(np.abs(slc_stack[a, p])**2, 1e-12))
        valid = power_db[np.isfinite(power_db)]
        limits = np.percentile(valid, [2, 98]) if valid.size else (-60, 0)
        im = ax.imshow(power_db, cmap="gray", vmin=limits[0], vmax=limits[1])
        ax.set_title(f"{name} (dB)")
        ax.set_axis_off()
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"Acquisition {a}: {chosen[a].id}", fontsize=10)
    fig.savefig(OUTPUT_FOLDER / f"polarizations_{a}.png", dpi=150)
    plt.show()


## 9. Pauli RGB for both acquisitions

Reuses your original Pauli convention **R = |HH−VV|², G = |HV+VH|², B = |HH+VV|²**. Each channel is locally averaged over 11×3 pixels then independently scaled for visual display. Red, green and blue are therefore qualitative representations, **not calibrated percentages**.

In [ ]:
from scipy.ndimage import uniform_filter

def make_pauli(slc, looks=(11, 3)):
    hh, hv, vh, vv = slc
    powers = [np.abs(hh-vv)**2, np.abs(hv+vh)**2, np.abs(hh+vv)**2]
    rgb = np.stack([uniform_filter(x.astype(np.float32), size=looks) for x in powers], axis=-1)
    for c in range(3):
        valid = rgb[..., c][np.isfinite(rgb[..., c])]
        scale = np.percentile(valid, 99) if valid.size else 1.0
        rgb[..., c] = np.clip(rgb[..., c] / max(float(scale), 1e-12), 0, 1)
    return np.nan_to_num(rgb, nan=0, posinf=1, neginf=0)

for a in range(slc_stack.shape[0]):
    rgb = make_pauli(slc_stack[a], PAULI_LOOKS)
    plt.figure(figsize=(8, 8))
    plt.imshow(rgb)
    plt.title(f"Pauli RGB — acquisition {a}\nR: HH−VV   G: HV+VH   B: HH+VV")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_FOLDER / f"pauli_rgb_{a}.png", dpi=180, bbox_inches="tight")
    plt.show()


## 10. Reopen the NetCDF stack


In [ ]:
ds2 = xr.open_dataset(OUTPUT_FOLDER / "biomass_stack.nc")
display(ds2)
HH_0 = ds2.slc_real.sel(acquisition=0, polarization="HH") + 1j * ds2.slc_imag.sel(acquisition=0, polarization="HH")
print("Reconstructed HH array shape:", HH_0.shape)
